In [10]:
import psycopg
import os
from dotenv import load_dotenv

In [31]:
dbname = os.getenv("DB_DATABASE")

load_dotenv()

try:
    with psycopg.connect(
        host=os.getenv("DB_HOST"),
        dbname=dbname,
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT"),
        connect_timeout=5
    ) as connection:

        with connection.cursor() as cursor:

#number of connections on database
            
            cursor.execute("""
                SELECT count(*)
                FROM pg_stat_activity;
            """)

            nr_connections = cursor.fetchone()[0]
            print('------------------------------------------------------------------------')
            print(f"Number of connections on database {dbname} is {nr_connections}.")

#maximal connections allowed
            
            cursor.execute("""
                SELECT setting 
                FROM pg_settings 
                WHERE name = 'max_connections';
            """)

            max_connections= cursor.fetchone()[0]
            print(f"Maximal connections allowed: {max_connections}.")
            print('------------------------------------------------------------------------')

#database size
            
            cursor.execute("""
                SELECT pg_size_pretty(pg_database_size(%s));
            """, (dbname,))

            database_size = cursor.fetchone()[0]
            print('------------------------------------------------------------------------')
            print('Size of database is:', database_size)
            print('------------------------------------------------------------------------')

#tables ordered by size
            
            cursor.execute("""
                SELECT relname AS table_name,
                pg_size_pretty(pg_total_relation_size(relid)) AS size
                FROM pg_catalog.pg_statio_user_tables
                ORDER BY pg_total_relation_size(relid) DESC;
            """)
            
            print(f"Tables ordered by size in database {dbname}:\n")
            for row in cursor.fetchall():
                print (row[0],row[1])
            print('------------------------------------------------------------------------')


           


except psycopg.OperationalError as e:
    print('Connection error:', e)
except psycopg.Error as e:
    print('Database error:', e)


------------------------------------------------------------------------
Number of connections on database pg4e_3fb3abd7f7 is 22.
Maximal connections allowed: 200.
------------------------------------------------------------------------
------------------------------------------------------------------------
Size of database is: 8797 kB
------------------------------------------------------------------------
Tables ordered by size in database pg4e_3fb3abd7f7:

tickets 96 kB
track_raw 64 kB
car_make 48 kB
car_model 48 kB
pg4e_meta 48 kB
course 40 kB
make 40 kB
roster 40 kB
student 40 kB
pg4e_debug 32 kB
pg4e_result 32 kB
automagic 24 kB
model 24 kB
car 24 kB
ages 8192 bytes
------------------------------------------------------------------------
